In [1]:
import os
import zipfile
from google.colab import files
import shutil

print("="*50)
print("STEP 1: UPLOAD YOUR ORL DATABASE (orl_faces.zip)")
print("="*50)

# Clean up previous runs
if os.path.exists('orl_faces.zip'):
    os.remove('orl_faces.zip')
if os.path.exists('/content/orl_faces'):
    shutil.rmtree('/content/orl_faces')
if os.path.exists('/content/orl_faces_temp'):
    shutil.rmtree('/content/orl_faces_temp')

# Upload the zip file
uploaded = files.upload()

if not uploaded:
    print("\nNo file uploaded. Please run the cell again and upload your zip file.")
else:
    zip_name = list(uploaded.keys())[0]
    print(f"\nSuccessfully uploaded: {zip_name}")

    # Unzip the file
    print(f"Unzipping {zip_name}...")
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall('/content/orl_faces_temp')

    # Find the correct nested directory (e.g., 'orl_faces' or 'att_faces')
    db_root = None
    for root, dirs, f in os.walk('/content/orl_faces_temp'):
        # 's1' and 's40' are the first and last folders in ORL
        if 's1' in dirs and 's4' in dirs:
            db_root = root
            break

    if db_root:
        # Move the found directory to the standard location
        shutil.move(db_root, '/content/orl_faces')
        # Only clean up '/content/orl_faces_temp' if it wasn't the root that was moved
        if db_root != '/content/orl_faces_temp' and os.path.exists('/content/orl_faces_temp'):
            shutil.rmtree('/content/orl_faces_temp') # Clean up temp
        print(f"Database successfully unzipped to '/content/orl_faces'")
    else:
        print("Could not find the main ORL directory (containing s1, s2, ... s40).")


STEP 1: UPLOAD YOUR ORL DATABASE (orl_faces.zip)


Saving DIP_OurDataset_PGM (1).zip to DIP_OurDataset_PGM (1).zip

Successfully uploaded: DIP_OurDataset_PGM (1).zip
Unzipping DIP_OurDataset_PGM (1).zip...
Database successfully unzipped to '/content/orl_faces'


In [2]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# --- Global variables to store the "trained" models ---
pca_model = None
lda_model = None
database_weights = None  # Weights of all images in the database
image_paths = []         # List of all image paths
image_labels = []        # List of labels (0-39) for each image
img_shape = None         # Shape of the images (e.g., 112, 92)

def compute_fisherfaces():
    global pca_model, lda_model, database_weights, image_paths, image_labels, img_shape

    DATABASE_PATH = '/content/orl_faces'
    if not os.path.exists(DATABASE_PATH):
        print("ERROR: Database path not found. Please run Cell 1 first.")
        return

    # --- Step 1: Load all images AND their labels ---
    face_vectors = []

    print("Loading database images and creating labels...")
    subject_label = 0
    for subject_folder in sorted(os.listdir(DATABASE_PATH)):
        folder_path = os.path.join(DATABASE_PATH, subject_folder)

        if os.path.isdir(folder_path):
            for img_name in sorted(os.listdir(folder_path)):
                if img_name.endswith('.pgm'):
                    img_path = os.path.join(DATABASE_PATH, subject_folder, img_name)
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

                    if img is not None:
                        # Ensure all images are of a consistent size (e.g., 112x92 for ORL)
                        expected_height, expected_width = 112, 92 # Standard ORL dimensions
                        if img.shape != (expected_height, expected_width):
                            # Resize image to the expected dimensions
                            img = cv2.resize(img, (expected_width, expected_height))

                        if img_shape is None:
                            img_shape = img.shape
                        face_vectors.append(img.flatten())
                        image_paths.append(img_path)
                        image_labels.append(subject_label)
            subject_label += 1

    if not face_vectors:
         print(f"No '.pgm' images found. Check the directory structure.")
         return

    face_matrix = np.array(face_vectors)
    image_labels = np.array(image_labels)

    print(f"Loaded {face_matrix.shape[0]} images from {subject_label} subjects.")
    print(f"Image shape: {img_shape} (Total pixels: {face_matrix.shape[1]}) ")

    # --- Step 2: Apply PCA for dimensionality reduction ---
    # This is the first step of Fisherfaces, to avoid the SSS problem
    # The expected ORL dataset has 400 images (n_samples) and 40 subjects (n_classes).
    # For Fisherfaces, n_pca_components is often set to n_samples - n_classes to ensure
    # the within-class scatter matrix for LDA is non-singular.
    # The original desired n_pca_components was 150 for a full dataset.

    if face_matrix.shape[0] <= subject_label:
        print("ERROR: Number of samples is not greater than the number of subjects. Cannot perform LDA effectively.")
        return

    dynamic_pca_components = max(1, face_matrix.shape[0] - subject_label)
    n_pca_components = min(dynamic_pca_components, 150, face_matrix.shape[0] - 1)

    print(f"\nApplying PCA to reduce dimensions to {n_pca_components} (adjusted dynamically based on data)...")
    pca_model = PCA(n_components=n_pca_components, whiten=True)
    pca_weights = pca_model.fit_transform(face_matrix)
    print("PCA reduction complete.")

    # --- Step 3: Apply LDA on the PCA-reduced data ---
    # This will find the best "separating" features
    # The number of components will be at most C-1 (subject_label - 1)
    print("Applying LDA to find best separating features (Fisherfaces)...")
    lda_model = LDA()

    # We fit the LDA model on the PCA-transformed data
    database_weights = lda_model.fit_transform(pca_weights, image_labels)

    print("Fisherfaces 'model' computed successfully.")
    print(f"  - PCA weights shape: {pca_weights.shape}")
    print(f"  - Final LDA weights (database) shape: {database_weights.shape}")


# Run the one-time computation
compute_fisherfaces()

Loading database images and creating labels...
Loaded 21 images from 4 subjects.
Image shape: (112, 92) (Total pixels: 10304) 

Applying PCA to reduce dimensions to 17 (adjusted dynamically based on data)...
PCA reduction complete.
Applying LDA to find best separating features (Fisherfaces)...
Fisherfaces 'model' computed successfully.
  - PCA weights shape: (21, 17)
  - Final LDA weights (database) shape: (21, 3)


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --- FIX 1: Correctly check if variables from Cell 2 exist ---
# We check if the *names* are in the global scope
if 'database_weights' not in globals() or 'image_paths' not in globals():
    print("Model not loaded! Please run Cell 2 first.")
else:
    # --- Create Interactive Widgets ---

    # Create a dropdown menu
    path_options = [p.replace('/content/orl_faces/', '') for p in image_paths]

    query_dropdown = widgets.Dropdown(
        options=path_options,
        description='Query Image:',
        style={'description_width': 'initial'},
        layout={'width': '500px'}
    )

    run_button = widgets.Button(
        description='Find Similar Faces',
        button_style='success',
        icon='search'
    )

    # Create an output area to display results
    output_area = widgets.Output()

    # --- Define the Query Function ---
    def find_and_display_matches(b):
        # Access global models and paths
        global pca_model, lda_model, database_weights, image_paths, img_shape

        # Clear previous results
        with output_area:
            clear_output()
            print("Processing query...")

        try:
            # --- 1. Get and process the selected query image ---
            query_short_path = query_dropdown.value
            QUERY_IMAGE_PATH = f'/content/orl_faces/{query_short_path}'

            query_img_gray = cv2.imread(QUERY_IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

            # --- FIX 2: Resize query image to match PCA model's input ---
            # The PCA model expects 10304 features, which is 92x112.
            # We must resize the query image to this shape.
            # cv2.resize expects (width, height)
            training_dims = (92, 112) # (width, height)
            query_img_resized = cv2.resize(query_img_gray, training_dims, interpolation=cv2.INTER_LANCZOS4)

            # Now, flatten the *resized* image
            query_vector = np.array(query_img_resized.flatten()).reshape(1, -1)

            # --- 2. Transform Query Image (PCA then LDA) ---
            # This will now work, as query_vector has 10304 features
            query_pca_weight = pca_model.transform(query_vector)
            query_final_weight = lda_model.transform(query_pca_weight)

            # --- 3. Calculate Euclidean Distances ---
            distances = []
            for i in range(database_weights.shape[0]):
                dist = np.linalg.norm(query_final_weight - database_weights[i])
                distances.append((dist, image_paths[i]))

            # --- 4. Sort and Get Top 4 Matches ---
            distances.sort(key=lambda x: x[0])
            top_matches = []
            for dist, path in distances:
                if path == QUERY_IMAGE_PATH: continue # Skip self
                if len(top_matches) < 4:
                    top_matches.append((dist, path))
                else:
                    break

            # --- 5. Display the Results ---
            with output_area:
                clear_output()
                print(f"Query: {query_short_path}")
                print("Results from Fisherfaces (PCA+LDA) method:")

                plt.figure(figsize=(18, 6))

                # Plot Query (use original for better quality display)
                plt.subplot(1, 5, 1)
                query_img_bgr = cv2.imread(QUERY_IMAGE_PATH)
                plt.imshow(cv2.cvtColor(query_img_bgr, cv2.COLOR_BGR2RGB))
                plt.title(f"Query Image\n({query_short_path})")
                plt.axis('off')

                # Plot Matches
                for i, (dist, path) in enumerate(top_matches):
                    plt.subplot(1, 5, i + 2)
                    match_img_bgr = cv2.imread(path)

                    # Format title
                    match_name = path.replace('/content/orl_faces/', '')
                    plt.title(f"Match {i+1} (Dist: {dist:.2f})\n{match_name}")

                    plt.imshow(cv2.cvtColor(match_img_bgr, cv2.COLOR_BGR2RGB))
                    plt.axis('off')

                plt.tight_layout()
                plt.show()

        except Exception as e:
            with output_area:
                clear_output()
                # Print a more detailed error message
                print(f"An error occurred: {e}")
                import traceback
                traceback.print_exc()

    # --- Link button to function and display ---
    run_button.on_click(find_and_display_matches)

    print("="*50)
    print("STEP 3: SELECT A QUERY IMAGE AND RUN")
    print("="*50)
    display(query_dropdown, run_button, output_area)

STEP 3: SELECT A QUERY IMAGE AND RUN


Dropdown(description='Query Image:', layout=Layout(width='500px'), options=('s1/ECE501_AU2340106_Cap.pgm', 's1…

Button(button_style='success', description='Find Similar Faces', icon='search', style=ButtonStyle())

Output()